#Intelligent Agents
## UC3 LLMs

##Lab 3.2: Starting Implicit and Explicit Reasoning with Language Models

In [ ]:
!pip install transformers torch accelerate

##Casual Language Models

### Example using a Native 'text-generation' Pipeline with GPT-2

Unlike Sequence-to-Sequence models (like FLAN-T5) which require encoder-decoder handling, Causal Language Models (like GPT-2) can directly use Hugging Face's built-in `'text-generation'` pipeline without any custom wrapper functions.

In [ ]:
from transformers import pipeline

# 1. Initialize the standard 'text-generation' pipeline using GPT-2
gpt_generator = pipeline("text-generation", model="openai-community/gpt2")

# 2. Define a prompt for the model to complete
#prompt = "The future of artificial intelligence in education is"
prompt = "If Alice is taller than Bob and Bob is taller than Carol, who is the shortest?"


# 3. Generate text directly using the pipeline
outputs = gpt_generator(prompt, max_length=50, num_return_sequences=1)

# 4. Display the generated text
print(outputs[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The future of artificial intelligence in education is not yet clear, but a few years from now the world will have a better sense of what is going on behind the scenes."

The researchers say they are not convinced that the new technology will lead to


## Sequence-to-Sequence Models

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline


#model_name = "openai-community/gpt2"
model_name = "google/flan-t5-base"  # Change to test other models
#model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

#generator=pipeline("text-generation",model=model, tokenizer = tokenizer)

# Due to issues with transformers 5.0.0 pipeline factory and direct import of Text2TextGenerationPipeline,
# we will create a custom generator function that directly uses the model's generate method.
def custom_generator(prompts, max_length=50):
    # 1. Standardizes input format to always handle batches (even if it is a single string)
    if isinstance(prompts, str):
        prompts = [prompts]

    # 2. Converts text prompts into numerical token IDs (tensors) that the model understands
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)

    # 3. Directly calls the model's native generate function to produce output tokens
    outputs = model.generate(**inputs, max_length=max_length)

    # 4. Decodes the predicted token IDs back into readable natural language text
    decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # 5. Formats the final output to match the list-of-dictionary structure returned by pipelines
    return [{'generated_text': text} for text in decoded_outputs]

generator = custom_generator


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

###Implicit Reasoning

In [ ]:
prompt = "If Alice is taller than Bob and Bob is taller than Carol, who is the shortest?"
result = generator(prompt, max_length=150)
print(result[0]['generated_text'])

Carol


###Explicit Reasoning: Encourage step-by-step reasoning

In [ ]:
prompt = """If Alice is taller than Bob and Bob is taller than Carol, explain step by step who is the shortest."""
result = generator(prompt, max_length=100)
print(result[0]['generated_text'])

Bob is taller than Carol. Bob is taller than Carol. Carol is the shortest person.


In [ ]:
prompt = "If Alice is taller than Bob and Bob is taller than Carol, who is the shortest, and why?"
result = generator(prompt, max_length=100)
print(result[0]['generated_text'])

Carol is taller than Bob
